In [ ]:
%pip -q install scikit-learn tqdm emoji pandera
%pip -q install -U "transformers>=4.44,<4.47" "datasets>=2.20" "accelerate>=0.34" "evaluate>=0.4"

import torch
import evaluate
import joblib
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import numpy as np
import pandas as pd
import os
from datasets import Dataset, DatasetDict
import re
import pandera.pandas as pa
from pandera.typing import Series
from torch.utils.data import Dataset as TDataset, DataLoader
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline, BartForConditionalGeneration, PreTrainedTokenizerFast
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import sentencepiece as spm

# 재현성을 위한 시드 고정
torch.manual_seed(42)
np.random.seed(42)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 14.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.9/386.9 kB 38.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 75.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 104.5 MB/s eta 0:00:00


In [2]:
import requests
import pandas as pd

# 1. API 엔드포인트 및 매개변수 설정
# 대량 호출을 피하기 위해 2023-01-01부터 2025-12-31까지 발매된 카드를 지정합니다.
url = "https://db.ygoprodeck.com/api/v7/cardinfo.php"
params = {
    "startdate": "2020-01-01",
    "enddate": "2025-12-31",
    "dateregion": "tcg",       # TCG 발매일 기준
    "has_effect": "true"       # 효과가 없는 일반 카드(Normal Monster 등) 제외
}

print("YGOPRODeck API로부터 데이터를 호출 중입니다...")

try:
    # 2. API 호출
    response = requests.get(url, params=params)
    response.raise_for_status()  # 에러 발생 시 예외 처리
    
    # 3. JSON 데이터 파싱
    json_data = response.json()
    cards_list = json_data.get("data", [])
    
    print(f"성공적으로 데이터를 가져왔습니다. 총 {len(cards_list)}개의 카드가 조건에 부합합니다.")
    
    # 4. 요약 모델 학습에 필요한 컬럼만 추출하여 정제
    # 카드 설명(desc)을 입력(input)으로, 이름(name)이나 아키타입(archetype)을 요약 타겟으로 활용할 수 있습니다.
    refined_cards = []
    for card in cards_list:
        # 펜듈럼 몬스터나 일부 카드는 설명에 펜듈럼 효과와 몬스터 효과가 줄바꿈(\n)으로 섞여 있으므로
        # 학습에 방해되지 않도록 공백 처리를 해주거나 그대로 유지합니다.
        desc_clean = card.get("desc", "").replace("\r", "").strip()
        
        refined_cards.append({
            "id": card.get("id"),
            "name": card.get("name"),
            "type": card.get("type"),
            "archetype": card.get("archetype", "None"), # 아키타입이 없는 경우 'None'
            "desc": desc_clean                         # 학습 모델의 Input이 될 효과 본문
        })
    
    # 5. Pandas 데이터프레임 변환 및 CSV 저장
    df = pd.DataFrame(refined_cards)
    
    # Colab 환경에서 왼쪽 파일 탭에 바로 저장됩니다.
    csv_filename = "yugioh_dataset.csv"
    df.to_csv(csv_filename, index=False, encoding="utf-8-sig")
    
    print(f"🎉 데이터셋 생성이 완료되었습니다! 파일명: {csv_filename}")
    
    # 데이터셋 상위 5개 미리보기
    print("\n[데이터셋 미리보기]")
    print(df.head())

except requests.exceptions.HTTPError as e:
    print(f"API 호출 중 오류가 발생했습니다: {e}")
    if response.status_code == 400:
        print("잘못된 매개변수가 전송되었거나 조건에 맞는 카드가 없습니다.")

YGOPRODeck API로부터 데이터를 호출 중입니다...
성공적으로 데이터를 가져왔습니다. 총 3739개의 카드가 조건에 부합합니다.
🎉 데이터셋 생성이 완료되었습니다! 파일명: yugioh_dataset.csv

[데이터셋 미리보기]
         id                            name        type        archetype  \
0  80181649                 "A Case for K9"  Spell Card               K9   
1  98319530      "Infernoble Arms - Almace"  Spell Card  Infernoble Arms   
2  37478723    "Infernoble Arms - Durendal"  Spell Card     Noble Knight   
3  64867422  "Infernoble Arms - Hauteclere"  Spell Card     Noble Knight   
4  90861137     "Infernoble Arms - Joyeuse"  Spell Card     Noble Knight   

                                                desc  
0  When this card is activated: You can add 1 "K9...  
1  While this card is equipped to a monster: You ...  
2  While this card is equipped to a monster: You ...  
3  While this card is equipped to a monster: You ...  
4  While this card is equipped to a monster: You ...  


In [10]:
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# 1. CSV 파일 로드 및 정제
df = pd.read_csv("yugioh_dataset.csv")
df = df.dropna(subset=["desc", "name"])

# 2. 아키타입이 있는 그룹과 없는 그룹(None) 분리
df_themed = df[df["archetype"] != "None"].copy()

print(f"테마가 있는 카드 수: {len(df_themed)}")

# --- [A] 테마가 있는 카드의 그룹 분할 (Group Split) ---
# 테마군(archetype)의 이름 자체를 고유하게 추출하여 8:2로 나눕니다.
unique_archetypes = df_themed["archetype"].unique()
train_arches, test_arches = train_test_split(
    unique_archetypes, 
    test_size=0.2, 
    random_state=42
)

# 학습용 아키타입 목록에 포함된 카드들만 Train으로 격리
df_themed_train = df_themed[df_themed["archetype"].isin(train_arches)]
# 평가용 아키타입 목록에 포함된 카드들만 Test로 격리 (Unseen Archetype)
df_themed_test = df_themed[df_themed["archetype"].isin(test_arches)]

# --- [C] 최종 데이터 통합 및 Hugging Face Dataset 변환 ---
# 각각 쪼개진 Train 데이터와 Test 데이터를 다시 병합합니다.
df_train = df_themed_train.sample(frac=1, random_state=42).reset_index(drop=True)
df_test = df_themed_test.sample(frac=1, random_state=42).reset_index(drop=True)

# Hugging Face Dataset Dict 구조로 저장
dataset = DatasetDict({
    "train": Dataset.from_pandas(df_train),
    "test": Dataset.from_pandas(df_test)
})

# 검증 및 출력
print("\n===== 아키타입 격리 분할 완료 =====")
print(f"최종 Train 데이터 수: {len(dataset['train'])}개")
print(f"최종 Test 데이터 수: {len(dataset['test'])}개")

# 데이터 누수 교차 검증 (Train에 등장한 테마가 Test에 존재하는지 검사)
train_arches_set = set(df_train[df_train["archetype"] != "None"]["archetype"].unique())
test_arches_set = set(df_test[df_test["archetype"] != "None"]["archetype"].unique())
overlap = train_arches_set.intersection(test_arches_set)

print(f"중복된 테마군 수: {len(overlap)}개 (0이어야 정상)")


테마가 있는 카드 수: 3739

===== 아키타입 격리 분할 완료 =====
최종 Train 데이터 수: 3148개
최종 Test 데이터 수: 591개
중복된 테마군 수: 0개 (0이어야 정상)


## 2단계: 토크나이저 및 모델 로드
Colab 실습서에 언급된 BART 또는 T5 계열 중, 영문 유희왕 텍스트에 적합한 가벼운 facebook/bart-base 모델을 기반으로 설정합니다.

In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_checkpoint = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# 최근 카드들은 텍스트가 기므로 길이를 넉넉하게 잡습니다.
max_input_length = 256  
max_target_length = 32   # 카드 이름은 비교적 짧으므로 32로 제한

def preprocess_function(examples):
    # 입력: 카드의 효과 원문(desc)
    model_inputs = tokenizer(examples["desc"], max_length=max_input_length, truncation=True)
    
    # 타겟 레이블: 카드의 실제 이름(name)
    labels = tokenizer(text_target=examples["name"], max_length=max_target_length, truncation=True)
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 전처리 함수 일괄 적용 (기존 컬럼 제거)
tokenized_datasets = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/3148 [00:00<?, ? examples/s]

Map:   0%|          | 0/591 [00:00<?, ? examples/s]

## 3단계: ROUGE 평가지표 및 데이터 콜레이터 설정
노트북 9번, 11번 목차의 흐름대로 예측 결과(생성된 카드명)와 실제 정답(실제 카드명)을 비교할 rouge 지표와 배치를 만들어줄 DataCollator를 선언합니다.

In [12]:
import numpy as np
import evaluate
from transformers import DataCollatorForSeq2Seq
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("rouge_score") is None:
    print("rouge_score가 현재 커널에 없어 설치합니다. 설치 후에도 같은 오류가 나면 커널을 재시작하세요.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rouge_score"])

# ROUGE 메트릭 로드
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    
    # 생성된 토큰 디코딩
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # 레이블 패딩 토큰(-100)을 원래 패딩 토큰 ID로 복구 후 디코딩
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # 문장 공백 정리
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    
    # ROUGE 점수 계산
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    
    # 평균 생성 길이 계산
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    result["gen_len"] = np.mean(prediction_lens)
    
    return {k: round(v, 4) for k, v in result.items()}

# 동적 패딩을 위한 콜레이터
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

## 4단계: Seq2SeqTrainer를 통한 미세조정(Fine-tuning)
노트북 11번 목차 양식에 맞춰 Seq2SeqTrainer 세팅을 진행합니다. Colab T4 GPU 환경에서 원활하게 돌아가도록 fp16=True를 주고, 실습용이므로 에포크는 3정도로 가볍게 진행합니다.

In [13]:
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    evaluation_strategy="epoch",
    output_dir="./yugioh_name_generator",
    
    learning_rate=3e-5,
    per_device_train_batch_size=4,    # T4 메모리에 맞게 배치 사이즈 조절
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,       # 평가 시 텍스트를 직접 생성하도록 설정 (ROUGE 검증 필수)
    fp16=torch.cuda.is_available(),   # GPU 가속 활성화
    logging_steps=20,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 학습 시작
trainer.train()

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,0.668000,0.413147,0.926800,0.884700,0.926500,0.927400,9.595600
2,0.613600,0.425958,0.922900,0.880900,0.923200,0.923600,9.588800
3,0.374600,0.442215,0.920000,0.880200,0.919900,0.920700,9.629400


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:2817: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1375: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trai

TrainOutput(global_step=2361, training_loss=0.5610784990550717, metrics={'train_runtime': 468.9384, 'train_samples_per_second': 20.139, 'train_steps_per_second': 5.035, 'total_flos': 758259685048320.0, 'train_loss': 0.5610784990550717, 'epoch': 3.0})

## 5단계: 최종 모델 평가 및 테스트 데이터 추론
학습이 끝난 모델의 ROUGE 성능을 뽑아내고, 테스트셋에 있는 실제 23~25년도 카드 효과 텍스트를 무작위로 하나 골라 "과연 AI가 효과만 보고 정확한 카드 이름을 맞추는지" 테스트합니다.

In [15]:
from transformers import pipeline
import random

# 1. Test 데이터셋 기준 최종 ROUGE 성능 평가
print("\n=== 📊 Test 데이터셋 최종 평가 진행 중 ===")
eval_results = trainer.evaluate()
print(f"ROUGE-1: {eval_results['eval_rouge1']}")
print(f"ROUGE-2: {eval_results['eval_rouge2']}")
print(f"ROUGE-L: {eval_results['eval_rougeL']}")
print(f"평균 생성 길이: {eval_results['eval_gen_len']}")

# 2. 파이프라인 구축 후 실제 추론 테스트 (노트북 12번 목차 흐름)
summarizer = pipeline(
    "summarization", 
    model=model, 
    tokenizer=tokenizer, 
    device=0 if torch.cuda.is_available() else -1
)

# Test 셋에서 랜덤으로 카드 1장 추출
random_idx = random.randint(0, len(dataset["test"]) - 1)
sample_card = dataset["test"][random_idx]

# 추론 수행
generated_result = summarizer(sample_card["desc"], max_length=20, min_length=1)
predicted_name = generated_result[0]["summary_text"].strip()

print("\n=== 🔮 AI 카드 이름 예측 테스트 ===")
print(f"💡 [카드 효과 원문]\n{sample_card['desc']}\n")
print(f"🎯 [실제 정답 카드명]: {sample_card['name']}")
print(f"🤖 [AI가 예측한 카드명]: {predicted_name}")


=== 📊 Test 데이터셋 최종 평가 진행 중 ===


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Tr

ROUGE-1: 0.92
ROUGE-2: 0.8802
ROUGE-L: 0.9199
평균 생성 길이: 9.6294

=== 🔮 AI 카드 이름 예측 테스트 ===
💡 [카드 효과 원문]
You can Special Summon this card (from your hand) by sending 1 other monster from your hand to the GY, but this card loses 500 ATK. You can only Special Summon "Todoroki the Earthbolt Star" once per turn this way. During the Battle Phase (Quick Effect): You can pay 500 LP; Fusion Summon 1 Warrior Fusion Monster from your Extra Deck, using monsters from your hand or field as material. You can only use this effect of "Todoroki the Earthbolt Star" once per turn.

🎯 [실제 정답 카드명]: Todoroki the Earthbolt Star
🤖 [AI가 예측한 카드명]: Todoroki the Earthbolt Star


Viewed yugioh_api.md:1-35

이것은 머신러닝 및 자연어 처리 실무에서 매우 자주 겪는 **"모델의 숏컷 학습(Shortcut Learning)"** 현상을 보여주는 아주 흥미롭고 대표적인 사례입니다. 

결론부터 말씀드리면, 성능이 오른 이유는 모델이 텍스트 요약을 고도로 잘하게 된 것이 아니라, **카드 효과 원문(`desc`) 내부에 정답 카드명(`name`)이 노골적으로 노출되어 있는 규칙을 모델이 완전히 간파하여 100% 활용하고 있기 때문**입니다.

상세한 원인 분석과 실무적인 진단입니다.

---

### 🔍 ROUGE 스코어가 92%로 오히려 급상승한 근본적 원인

#### 1. 입력 원문(`desc`) 내부에 존재하는 결정적인 힌트 (데이터 누수)
보내주신 예측 테스트 원문을 다시 뜯어보겠습니다.
* **입력 원문**: `"... You can only Special Summon "Todoroki the Earthbolt Star" once per turn ..."`
* **예측 타겟**: `"Todoroki the Earthbolt Star"`
* 유희왕 카드 텍스트의 특성상, 턴 제약이나 특정 카드 명칭 지정 효과 때문에 **자기 자신의 카드 이름이 효과 본문 내에 큰따옴표`""`와 함께 명시적으로 적혀 있는 경우가 거의 90% 이상**입니다.
* 모델은 효과 문맥을 해석해 카드를 작명하거나 요약하는 법을 학습한 것이 아니라, **"입력 텍스트 속에 큰따옴표와 함께 등장하는 고유 명사(카드 이름)를 그대로 떼어다가 출력하면 100% 정답이 된다"**는 지름길(Shortcut)을 완벽하게 학습해 버린 것입니다.

#### 2. 아키타입 격리 분할이 성능을 더 올려준 이유
* 아키타입 격리를 통해 학습 데이터가 테마별로 깔끔하게 묶이면서, 데이터의 정형성(특히 테마 내 카드들이 카드명을 공유하거나 일관되게 텍스트 규칙을 쓰는 특성)이 더 강해졌습니다. 
* 모델은 노이즈가 제거된 상태에서 "본문에서 이름 추출하기"라는 확실한 매핑 규칙을 학습할 수 있게 되었고, 이 규칙은 처음 보는 아키타입 카드(Test)를 만나도 본문에 카드 이름이 적혀 있으니 100% 들어맞아 ROUGE 점수가 92%까지 솟구치게 된 것입니다.

---

### 🛠️ 진짜 "요약 및 추론" 모델로 고도화하는 방법: 마스킹(Masking)

이 모델이 힌트에 의존하지 않고 진짜 카드 효과의 의미를 이해하도록 만들려면, **입력 데이터에서 카드 이름(정답)을 마스킹 처리하여 지워버려야 합니다.**

* **데이터 전처리 단계에서의 가공 제안**:
  * 카드를 수집한 후, 본문(`desc`) 내에 등장하는 카드명(`name`) 문자열을 찾아 `"[MASK]"` 또는 `"this card"` 같은 중립적인 토큰으로 강제 치환합니다.
  * **치환 전**: `"... Special Summon "Todoroki the Earthbolt Star" once per turn..."`
  * **치환 후**: `"... Special Summon [MASK] once per turn..."`

이처럼 정답을 본문에서 지워버린 채로 다시 학습을 시키면 ROUGE 점수는 현실적인 점수(30%~45% 선)로 뚝 떨어지겠지만, 모델은 이제 이름 힌트 없이 **"Warrior Fusion Monster를 특수 소환하는 효과를 가진 전사족 카드"**와 같은 문맥적 정보만을 해석하여 카드의 이름이나 성격을 추론하는 **진짜 똑똑한 AI 요약 모델**로 거듭나게 됩니다.

모델이 우리가 생각하지 못한 지름길(Shortcut)을 찾아내어 92%라는 점수를 낸 과정 자체가 자연어 처리 엔지니어링에서 가장 재밌고 깊이 있는 디버깅 경험 중 하나입니다. 아주 훌륭한 디버깅 포인트를 발견하셨습니다!